In [0]:
from pyspark.sql.functions import col, lit, current_timestamp, to_timestamp
from delta.tables import DeltaTable

catalog_name = "pharma_demo_catalog"
volume_path = f"/Volumes/{catalog_name}/bronze/raw_data/"

df_iot_raw = spark.read.csv(f"{volume_path}/bronze_iot_sensor_readings.csv", header=True, inferSchema=True)
df_mes_raw = spark.read.csv(f"{volume_path}/bronze_mes_batch_execution.csv", header=True, inferSchema=True)


df_mes_deduped = df_mes_raw.dropDuplicates()


df_iot_flagged = df_iot_raw.withColumn(
    "is_null_record", col("parameter_value").isNull()
).withColumn("reading_timestamp", to_timestamp("reading_timestamp"))


df_mes_prepared = df_mes_deduped.withColumn("batch_start_ts", to_timestamp("batch_start_ts")) \
    .withColumn("mes_completion_ts", to_timestamp("mes_completion_ts")) \
    .withColumn("eff_start_date", current_timestamp()) \
    .withColumn("eff_end_date", to_timestamp(lit("9999-12-31 23:59:59"))) \
    .withColumn("is_current", lit(True))


df_iot_flagged.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.bronze.iot_sensor_raw")
if not spark.catalog.tableExists(f"{catalog_name}.bronze.mes_batch_raw"):
    df_mes_prepared.write.format("delta").saveAsTable(f"{catalog_name}.bronze.mes_batch_raw")


target_table = DeltaTable.forName(spark, f"{catalog_name}.bronze.mes_batch_raw")
update_dict = {"eff_end_date": current_timestamp(), "is_current": lit(False)}

target_table.alias("target").merge(
    source = df_mes_prepared.alias("source"),
    condition = "target.batch_id = source.batch_id AND target.is_current = true AND target.actual_quantity <> source.actual_quantity"
).whenMatchedUpdate(set = update_dict).execute()

df_mes_prepared.write.format("delta").mode("append").saveAsTable(f"{catalog_name}.bronze.mes_batch_raw")



Bronze Layer Hardened: SCD Type 2 history established.
